In [ ]:
import pygame, math, numpy as np

LARGURA, ALTURA = 900, 650

class BraitenbergRobot:
    def __init__(self, x, y):
        self.x, self.y, self.theta = float(x), float(y), 0.0
        self.sensor_angles = [-math.pi/2, -math.pi/4, 0.0, math.pi/4, math.pi/2]
        self.sensor_range = 200.0
        self.sensor_readings = [self.sensor_range] * 5

    def cast_rays(self, obstacles):
        self.sensor_readings = []
        for beta in self.sensor_angles:
            angle = self.theta + beta
            min_dist = self.sensor_range
            for step in range(5, int(self.sensor_range), 4):
                rx = self.x + step * math.cos(angle)
                ry = self.y + step * math.sin(angle)
                if rx <= 0 or rx >= LARGURA or ry <= 0 or ry >= ALTURA:
                    min_dist = float(step)
                    break
                if any(obs.collidepoint(rx, ry) for obs in obstacles):
                    min_dist = float(step)
                    break
            self.sensor_readings.append(min_dist)

    def update_braitenberg(self):
        v_cruzeiro = 2.0
        L = 30.0 
        
        dist_esq = min(self.sensor_readings[0], self.sensor_readings[1])
        dist_dir = min(self.sensor_readings[3], self.sensor_readings[4])
        dist_centro = self.sensor_readings[2]
        
        if dist_centro < 50.0:
            v_L, v_R = -v_cruzeiro, v_cruzeiro
        else:
            ganho = 50.0
            v_L = v_cruzeiro + (ganho / dist_dir) if dist_dir < 150 else v_cruzeiro
            v_R = v_cruzeiro + (ganho / dist_esq) if dist_esq < 150 else v_cruzeiro
            
        v = (v_L + v_R) / 2.0
        w = (v_R - v_L) / L
        
        self.theta += w
        self.x += v * math.cos(self.theta)
        self.y += v * math.sin(self.theta)

def main_lab4():
    pygame.init()
    screen = pygame.display.set_mode((LARGURA, ALTURA))
    robot = BraitenbergRobot(100, 100)
    obstacles = [pygame.Rect(300, 100, 100, 400), pygame.Rect(600, 200, 100, 100)]
    
    running = True
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT: running = False
            
        robot.cast_rays(obstacles)
        robot.update_braitenberg()
        
        screen.fill((20, 24, 30))
        for obs in obstacles: pygame.draw.rect(screen, (180, 50, 50), obs)
        pygame.draw.circle(screen, (0, 200, 255), (int(robot.x), int(robot.y)), 16)
        
        for i, beta in enumerate(robot.sensor_angles):
            angle = robot.theta + beta
            rx = robot.x + robot.sensor_readings[i] * math.cos(angle)
            ry = robot.y + robot.sensor_readings[i] * math.sin(angle)
            pygame.draw.line(screen, (0, 255, 100), (int(robot.x), int(robot.y)), (int(rx), int(ry)), 1)

        pygame.display.flip()
        pygame.time.Clock().tick(60)
    pygame.quit()

if __name__ == "__main__": main_lab4()

pygame 2.6.1 (SDL 2.28.4, Python 3.10.12)
Hello from the pygame community. https://www.pygame.org/contribute.html
